Anthropic SDK 是由 AI 研究公司 Anthropic 官方提供的一套软件开发工具包（Software Development Kit），主要用于帮助开发者更轻松地将 Anthropic 旗下的 Claude 系列大模型（如 Claude 3.5 Sonnet, Claude 3 Opus 等）集成到自己的应用程序、服务或工作流中。

In [1]:
pip install anthropic

Note: you may need to restart the kernel to use updated packages.


核心概念全览
## 1. 同步 vs 异步客户端

In [6]:
import anthropic

# ── 同步客户端：普通 .py 脚本用这个 ──────────────────────
client = anthropic.Anthropic(api_key="sk-ant-...")
# 或者不传 api_key，SDK 自动读取环境变量 ANTHROPIC_API_KEY

# ── 异步客户端：FastAPI / Jupyter 用这个 ─────────────────
async_client = anthropic.AsyncAnthropic()

In [8]:
import anthropic
import os
os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"

client = anthropic.Anthropic()

# client.messages.create() 是最核心的方法
# 每次调用就是发一条消息，等待完整回复
response = client.messages.create(
    model="claude-sonnet-4-6",      # 使用的模型
    max_tokens=1024,                 # 最多生成多少 token（必填）
    messages=[                       # 对话历史（列表格式）
        {"role": "user", "content": "什么是 RAG？"}
    ]
)

# 从响应对象里取出文本
print(response.content[0].text)

# 响应对象还包含：
print(response.model)           # 实际使用的模型名
print(response.stop_reason)     # 停止原因：end_turn / max_tokens
print(response.usage)           # token 用量：input_tokens / output_tokens

# RAG（检索增强生成）

## 基本概念

**RAG（Retrieval-Augmented Generation，检索增强生成）** 是一种将**信息检索**与**大语言模型生成**相结合的 AI 技术架构。

---

## 核心思想

> 让模型在回答问题时，先从外部知识库中**检索相关信息**，再基于检索结果**生成回答**。

---

## 工作流程

```
用户提问
   ↓
① 检索阶段：将问题转为向量，在知识库中搜索相关文档
   ↓
② 增强阶段：将检索到的相关内容拼接到提示词中
   ↓
③ 生成阶段：LLM 基于上下文生成最终回答
   ↓
输出答案
```

---

## 为什么需要 RAG？

| 问题 | RAG 的解决方式 |
|------|-------------|
| 模型知识有截止日期 | 可接入实时/最新知识库 |
| 模型容易"幻觉" | 基于真实文档生成，更可靠 |
| 私有数据无法训练 | 无需微调，直接检索私有文档 |
| 训练成本高 | 更新知识库即可，无需重新训练 |

---

## 核心组件

```
知识库构建
├── 文档加载（PDF、Word、网页等）
├── 文本分割（Chunking）
├── 向量化（Embedding）
└── 向量数据库存储（Faiss、Pinecone、Milvus等）

检索系统
├── 稠密检索（语义搜索）
├── 稀疏检索（BM25关键词）
└── 混合检索

生成模型
└── GPT、Claude、LLaMA 等 LLM
```

---

## 典型应用场景

- 📚 **企业知识库问答**（内部文档检索）
- 🔬 **学术研究助手**（论文检索与总结）
- 💬 **客服系统**（产品手册问答）
- ⚖️ **法律/医疗咨询**（专业文档查询）

---

## 简单类比

> RAG 就像一个**开卷考试的学生**：
> - 普通 LLM = 闭卷考试（只靠记忆）
> - RAG = 开卷考试（可以查资料再作答）

---

需要深入了解某个具体方面（如向量数据库、Embedding、具体实现框架等）吗？
claude-sonnet-4-6
end_turn
Usage(cache_creation=CacheCreation(ephemeral_1h

## 2. 最基础的调用

In [9]:
import anthropic

client = anthropic.Anthropic()

# client.messages.create() 是最核心的方法
# 每次调用就是发一条消息，等待完整回复
response = client.messages.create(
    model="claude-sonnet-4-6",      # 使用的模型
    max_tokens=1024,                 # 最多生成多少 token（必填）
    messages=[                       # 对话历史（列表格式）
        {"role": "user", "content": "什么是 RAG？"}
    ]
)

# 从响应对象里取出文本
print(response.content[0].text)

# 响应对象还包含：
print(response.model)           # 实际使用的模型名
print(response.stop_reason)     # 停止原因：end_turn / max_tokens
print(response.usage)           # token 用量：input_tokens / output_tokens

# RAG（检索增强生成）

## 基本概念

**RAG（Retrieval-Augmented Generation，检索增强生成）** 是一种将**信息检索**与**大语言模型生成**相结合的 AI 技术框架。

---

## 核心思想

> 让 AI 在回答问题前，先去"查资料"，再基于查到的内容生成答案。

---

## 工作流程

```
用户提问
   ↓
① 检索（Retrieval）
   将问题转换为向量，在知识库中搜索相关文档
   ↓
② 增强（Augmented）
   将检索到的相关内容作为上下文
   ↓
③ 生成（Generation）
   LLM 结合上下文生成最终答案
```

---

## 为什么需要 RAG？

| 问题 | RAG 解决方案 |
|------|------------|
| 模型知识有截止日期 | 可接入实时/最新数据 |
| 模型可能"幻觉"（胡编） | 基于真实文档生成，更可靠 |
| 无法使用私有数据 | 可接入企业内部知识库 |
| 重新训练成本高 | 无需微调，更新知识库即可 |

---

## 技术组成

```
知识库构建阶段：
文档 → 分块（Chunking）→ 向量化（Embedding）→ 存入向量数据库

查询阶段：
问题 → 向量化 → 相似度搜索 → 召回相关片段 → 送入 LLM
```

---

## 典型应用场景

- 📚 **企业知识库问答**（内部文档、FAQ）
- 🔍 **智能客服**
- 📄 **文档分析与问答**
- 🏥 **医疗/法律专业领域助手**
- 💻 **代码库问答**

---

## 简单类比

> RAG 就像一个**开卷考试的学生**：
> - 普通 LLM = 闭卷考试（只靠记忆）
> - RAG = 开卷考试（可以翻书查资料再作答）

---

如果你想深入了解某个具体环节（如向量数据库、Embedding、实现框架等），可以继续提问！
claude-sonnet-4-6
end_turn
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=

## 3. system 参数（系统提示词）

In [10]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    # system 定义模型的角色和行为规则
    # 相当于在对话开始前给模型的"工作说明"
    system="你是一个专注于 AI 领域的技术助手，回答简洁专业，不超过 100 字。",
    messages=[
        {"role": "user", "content": "解释一下 Transformer"}
    ]
)

## 4. 多轮对话（手动维护历史）

In [11]:
# SDK 本身无状态，多轮对话需要自己维护消息列表
# 每次请求都把完整历史传进去

history = []

def chat(user_message: str) -> str:
    # 把用户消息加入历史
    history.append({"role": "user", "content": user_message})

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system="你是一个 AI 学习助手",
        messages=history,            # 传完整历史
    )

    reply = response.content[0].text

    # 把模型回复也加入历史，供下一轮使用
    history.append({"role": "assistant", "content": reply})

    return reply

# 连续对话，模型记得上下文
print(chat("什么是 RAG？"))
print(chat("它有哪些主要组件？"))   # 模型知道"它"指 RAG
print(chat("给我一个 Python 实现思路"))

# RAG（检索增强生成）

## 定义

**RAG（Retrieval-Augmented Generation，检索增强生成）** 是一种将**信息检索**与**大语言模型生成**相结合的 AI 技术框架。

---

## 核心思想

```
用户提问 → 检索相关文档 → 将文档 + 问题一起输入 LLM → 生成答案
```

---

## 为什么需要 RAG？

| 问题 | 说明 |
|------|------|
| 🕒 知识截止 | LLM 训练数据有时间限制，无法获取最新信息 |
| 🏢 私有数据 | 企业内部数据未包含在 LLM 训练集中 |
| 🎭 幻觉问题 | LLM 可能"编造"不存在的信息 |
| 💰 成本问题 | 频繁微调模型成本极高 |

---

## 工作流程

```
1. 索引阶段（离线）
   文档 → 切分 → Embedding（向量化）→ 存入向量数据库

2. 检索阶段（在线）
   用户问题 → Embedding → 向量相似度搜索 → 取出 Top-K 文档

3. 生成阶段
   [系统提示 + 检索到的文档 + 用户问题] → LLM → 最终答案
```

---

## 关键组件

- **文档加载器**：PDF、网页、数据库等
- **文本分割器**：将长文档切成小块（Chunk）
- **Embedding 模型**：将文本转为向量
- **向量数据库**：如 Pinecone、Chroma、Faiss、Milvus
- **LLM**：GPT、Claude、Llama 等

---

## 简单示例

```python
# 使用 LangChain 实现简单 RAG
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import RetrievalQA

# 1. 创建向量数据库
vectordb = Chroma.from_documents(docs, OpenAIEmbeddings())

# 2. 创建检索链
qa_chain = RetrievalQA.from_chain_type(
    llm=

## 5. 流式输出

In [12]:
# 普通调用：等模型生成完才返回（可能要等 5-10 秒）
# 流式调用：每生成一个 token 就立刻推过来（打字机效果）

# ── 同步流式 ─────────────────────────────────────────────
with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[{"role": "user", "content": "介绍一下 Agent 架构"}],
) as stream:
    for text in stream.text_stream:
        # end="" 不换行，flush=True 立刻输出（不缓冲）
        print(text, end="", flush=True)

print()  # 最后换行

# ── 异步流式（FastAPI 里用）───────────────────────────────
async def stream_response(prompt: str):
    async with async_client.messages.stream(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        async for text in stream.text_stream:
            print(text, end="", flush=True)

# Agent 架构介绍

## 什么是 Agent？

Agent（智能体）是一种**能够感知环境、自主决策并采取行动**以完成目标的 AI 系统。与简单的问答模型不同，Agent 具备**规划、记忆、工具使用**等能力。

---

## 核心架构组成

```
┌─────────────────────────────────────────┐
│                  Agent                   │
│                                         │
│  ┌──────────┐      ┌──────────────────┐ │
│  │  感知层   │ ───▶ │    大脑/推理核心  │ │
│  │ (Input)  │      │   (LLM/Policy)   │ │
│  └──────────┘      └────────┬─────────┘ │
│                             │           │
│  ┌──────────┐      ┌────────▼─────────┐ │
│  │  记忆层   │ ◀──▶ │    规划/决策层    │ │
│  │ (Memory) │      │   (Planning)     │ │
│  └──────────┘      └────────┬─────────┘ │
│                             │           │
│                    ┌────────▼─────────┐ │
│                    │     行动层        │ │
│                    │  (Action/Tools)  │ │
│                    └──────────────────┘ │
└─────────────────────────────────────────┘
```

---

## 四大核心模块

### 1. 🧠 推理核心（Brain）
- 通常由 **LLM**（如 GPT-4、Claude）充当
- 负责理解任务、推理分析、

## 6. Tool Use（工具调用）—— Agent 的核心
这是构建 Agent 最关键的功能：让模型决定何时调用外部工具。

In [13]:
import json

# ── 第一步：定义工具（告诉模型有哪些工具可用）────────────
tools = [
    {
        "name": "get_weather",
        "description": "查询指定城市的当前天气",  # 描述越清晰，模型调用越准确
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "城市名称，如：北京、上海",
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "温度单位，默认摄氏度",
                },
            },
            "required": ["city"],  # city 是必填参数
        },
    },
    {
        "name": "search_web",
        "description": "搜索互联网获取最新信息",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "搜索关键词"},
            },
            "required": ["query"],
        },
    },
]

# ── 第二步：发送请求，模型决定是否调用工具 ───────────────
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    tools=tools,
    messages=[{"role": "user", "content": "北京今天天气怎么样？"}],
)

# ── 第三步：判断模型是否要调用工具 ──────────────────────
if response.stop_reason == "tool_use":
    # 模型决定调用工具，从响应里提取调用信息
    tool_call = next(b for b in response.content if b.type == "tool_use")

    print(f"模型要调用工具：{tool_call.name}")    # get_weather
    print(f"传入参数：{tool_call.input}")         # {"city": "北京"}

    # ── 第四步：执行真正的工具函数 ──────────────────────
    def get_weather(city: str, unit: str = "celsius") -> dict:
        # 实际项目里这里调用真实天气 API
        return {"city": city, "temperature": 22, "condition": "晴天", "unit": unit}

    tool_result = get_weather(**tool_call.input)

    # ── 第五步：把工具结果返回给模型，让它生成最终回答 ──
    final_response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        tools=tools,
        messages=[
            {"role": "user", "content": "北京今天天气怎么样？"},
            {"role": "assistant", "content": response.content},  # 模型上一次的回复
            {
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_call.id,           # 必须对应
                        "content": json.dumps(tool_result),    # 工具执行结果
                    }
                ],
            },
        ],
    )
    print(final_response.content[0].text)
    # 输出：北京今天天气晴朗，气温 22°C，适合外出。

模型要调用工具：get_weather
传入参数：{'city': '北京'}
今天北京的天气情况如下：

- 🌤️ **天气状况**：晴天
- 🌡️ **当前温度**：22°C

天气不错，适合外出活动！记得注意防晒哦 😊


## 7. 错误处理

In [14]:
try:
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        messages=[{"role": "user", "content": "你好"}],
    )
except anthropic.AuthenticationError:
    # API Key 无效或未设置
    print("API Key 错误，检查 ANTHROPIC_API_KEY 环境变量")

except anthropic.RateLimitError:
    # 触发请求限速（请求太频繁）
    print("请求太频繁，等待几秒后重试")

except anthropic.BadRequestError as e:
    # 请求参数有问题（余额不足、模型名称错误等）
    print(f"请求参数错误：{e}")

except anthropic.APIConnectionError:
    # 网络连接失败
    print("网络连接失败，检查网络或代理设置")

## 8. 完整速查表

In [17]:
import anthropic

client = anthropic.AsyncAnthropic()

# ── 基础调用 ──────────────────────────────────────────────
response = await client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="你是一个简洁的助手",
    messages=[{"role": "user", "content": "什么是 RAG？"}],
)

text = response.content[0].text
print(text)

input_tokens  = response.usage.input_tokens
output_tokens = response.usage.output_tokens
print(f"Token 用量：输入 {input_tokens}，输出 {output_tokens}")

# ── 流式输出 ──────────────────────────────────────────────
async with client.messages.stream(          # async with，不是 with
    model="claude-sonnet-4-6",
    max_tokens=1024,
    messages=[{"role": "user", "content": "用一句话解释 Transformer"}],
) as stream:
    async for text in stream.text_stream:   # async for，不是 for
        print(text, end="", flush=True)
print()

# RAG（检索增强生成）

**RAG（Retrieval-Augmented Generation）** 是一种将**信息检索**与**大语言模型生成**相结合的技术框架。

---

## 核心思路

```
用户提问 → 检索相关文档 → 将文档 + 问题输入 LLM → 生成回答
```

---

## 主要组成部分

| 组件 | 作用 |
|------|------|
| **知识库** | 存储外部文档/数据 |
| **检索器** | 找出与问题相关的内容（常用向量相似度） |
| **生成器** | 基于检索结果由 LLM 生成最终答案 |

---

## 解决了什么问题？

- ✅ **知识过时**：LLM 训练数据有截止日期，RAG 可接入最新数据
- ✅ **幻觉问题**：有据可查，减少模型"编造"内容
- ✅ **私有数据**：可接入企业内部文档
- ✅ **可追溯**：答案来源可引用

---

## 典型应用场景

- 企业内部知识库问答
- 客服系统
- 文档智能搜索
- 医疗/法律等专业领域问答

---

## 简单类比

> 就像开卷考试：LLM 是"大脑"，检索系统是"参考书"，答题时先查书再作答，比纯靠记忆更准确。
Token 用量：输入 26，输出 458
**Transformer 是一种完全基于「自注意力机制」的深度学习模型，能够并行地捕捉序列中任意位置之间的依赖关系，从而高效处理自然语言等序列数据。**
